# 第 3 章｜Tool Calling

依序執行每一格；可修改標示的參數後重跑。

## 執行前：可替換設定總覽

以下項目都可以依測試環境替換：

- `.env` Provider：正式課程填 `CILLM_API_KEY`；只有個人本機測試才填 `OPENAI_API_KEY`；兩者都有時 CILLM 優先。
- CILLM：`CILLM_BASE_URL`、`CILLM_USER_ID`、`CILLM_PLATFORM`、`CILLM_AGENT`、`GPT_OSS_MODEL_NAME`、`GEMMA_MODEL_NAME`、`CILLM_VISION_FORMAT`。
- CILLM 圖片：預設 `CILLM_VISION_FORMAT=html`，由 Gemma 4 31B IT 直接處理；後端支援多模態 content blocks 後可改成 `blocks`。
- OpenAI：`OPENAI_API_KEY`、`OPENAI_MODEL_NAME`；預設 `gpt-4o`，僅供個人筆電模擬測試。
- 密碼式 AES：第 6 章由使用者在 Notebook 隱藏輸入設定保險庫密碼；密碼不寫入 `.env` 或加密檔。
- 路徑：只有從其他工作目錄啟動 Notebook 時才需要調整 `ROOT`；一般從教材根目錄或 `notebooks/` 啟動不必修改。

> 請勿把含有真實 Key 的 `.env`、Notebook 輸出或截圖提交到 Git。

### 本章可替換

- `USER_REQUEST`：每個 Tool 範例的問題都可替換。
- `MAX_TOKENS`：替換路由判斷或回答的最大輸出 token 數。
- `math_tool(a, op, b)`：替換兩個數值與 `+ - * /` 運算子。
- 圖片路徑與圖片問題：替換 `mock_boarding_pass.png` 及傳給 `analyze_image` 的問題。
- Excel 路徑：替換 `flight_delays.xlsx`。
- `GENERATED_CODE`：替換受限 pandas 程式；必須設定 `result`，且不可 import、存取網路、系統或任意檔案。
- `TOOL_REGISTRY`：可在 `course_utils.py` 新增 Tool 名稱、描述與函式。

In [34]:
from pathlib import Path
import importlib, os, sys
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks": ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import course_utils
importlib.reload(course_utils)
from course_utils import *
print("教材根目錄：", ROOT)
connection = verify_cillm_key()
print("✅ API Key 驗證成功")
print("目前 Provider：", connection["provider"])
print("目前模型：", connection["model"])
print("首次回覆：", connection["reply"])

教材根目錄： C:\Users\rathe\Project\cillm\CILLM_Workshop\Lecture03
✅ API Key 驗證成功
目前 Provider： openai
目前模型： gpt-4o
首次回覆： CILLM 連線成功


## 檢查目前 API Key 的基本 Scope

下方 Cell 會查詢並列出這支 Key 實際具備的 RBAC scopes。本教材呼叫 GPT-OSS 至少需要 `llm.chat`。

In [35]:
scopes = get_current_key_scopes()
if scopes is None:
    print("目前使用 OpenAI API；CILLM RBAC scope 不適用。")
else:
    print("目前 CILLM_API_KEY 具備的 scopes：")
    for scope in scopes:
        print("-", scope)
    required_scope = "llm.chat"
    print(f"✅ 已具備教材基本 scope：{required_scope}" if "*" in scopes or required_scope in scopes else f"❌ 缺少教材基本 scope：{required_scope}")

目前使用 OpenAI API；CILLM RBAC scope 不適用。


> **執行模式 Hint**
>
> - 本教材每章第一格會取得 `CILLM_API_KEY`；若 `.env` 未設定，Notebook 會以隱藏輸入提示使用者填入。
> - 取得 Key 後會立即呼叫一次 `openai/gpt-oss-120b`，確認 Key 與連線可用。
> - `CILLM_BASE_URL` 預設沿用 Lecture 02 的測試端點；缺少或無效 Key 時會立即停止。
> - 正式課程的 CILLM 模式：文字使用 GPT-OSS，圖片使用 NVIDIA NIM 的 `google/gemma-4-31b-it`，不會 fallback 到 OpenAI。
> - `CILLM_VISION_FORMAT=html` 會把 Base64 圖片包成 NVIDIA NIM 支援的 `<img src="data:...">` 字串，適合目前只接受字串 content 的 gateway。
> - CILLM gateway 部署支援多模態陣列的 schema 後，可改成 `CILLM_VISION_FORMAT=blocks`，送出標準 OpenAI `image_url` content blocks。
> - 只有 OpenAI Key 時，才以 `gpt-4o` 模擬文字與圖片流程，供個人筆電測試。

> **Tool Router 流程**
>
> 1. 程式將 `TOOL_REGISTRY` 的 `name`、`description` 與參數範例交給 AI。
> 2. AI 只能依 description 選擇 Tool，並回傳 JSON。
> 3. LangChain 直接產生 `ToolRouteResult` Pydantic instance 並驗證路由結果，再由各 Tool 的 Pydantic Arguments Model 驗證參數。
> 4. 通過驗證後才執行 Tool；`none` 代表不需要 Tool。

## Tool Registry：AI 實際用來判斷的 Description

In [36]:
registry_is_current = True
for name, meta in TOOL_REGISTRY.items():
    print(f"\n{name}")
    print("  description:", meta.get("description", "(未提供)"))
    arguments_example = meta.get("arguments_example")
    if arguments_example is None:
        registry_is_current = False
        arguments_example = "(舊版 Registry 未提供；請先重新執行最前面的初始化 Cell)"
    print("  arguments example:", arguments_example)

if not registry_is_current:
    print("\n⚠️ 偵測到 Kernel 中的舊版 TOOL_REGISTRY，請重新執行第一個初始化 Cell。")
else:
    print("\nPydantic route schema:")
    show(ToolRouteResult.model_json_schema())


math_tool
  description: 需要精確執行兩個數值的加、減、乘、除時使用；arguments 必須包含 a、op、b。
  arguments example: {'a': 312, 'op': '*', 'b': 0.87}

image_tool
  description: 問題必須查看圖片像素、辨識圖片文字或理解視覺內容時使用；arguments 必須包含 question。
  arguments example: {'question': '圖片中的航班與登機門是什麼？'}

excel_python_tool
  description: 需要對 Excel 表格做精確篩選、排序、統計、分組或聚合時使用；arguments 必須包含 question。
  arguments example: {'question': '依部門計算平均延誤'}

Pydantic route schema:
{
  "properties": {
    "tool_name": {
      "description": "選擇 TOOL_REGISTRY 中的一個工具名稱；不需要工具時填 none",
      "title": "Tool Name",
      "type": "string"
    },
    "reason": {
      "default": "模型未提供選擇理由",
      "description": "根據工具 description 說明選擇理由",
      "title": "Reason",
      "type": "string"
    },
    "arguments": {
      "additionalProperties": true,
      "description": "傳給工具的參數",
      "title": "Arguments",
      "type": "object"
    }
  },
  "required": [
    "tool_name"
  ],
  "title": "ToolRouteResult",
  "type": "object"
}


## 一次一題體驗 Description-based Routing

保留一個 `USER_REQUEST` 啟用；要測其他情境時，先註解目前題目，再取消另一題的註解。

In [37]:
# 預設：數學 Tool
USER_REQUEST = "312 個座位、載客率 87%，精確計算旅客數。"

# 圖片 Tool：請先註解上面的 USER_REQUEST，再取消下行註解
USER_REQUEST = "這張登機證的航班和登機門是什麼？"

# Excel Tool
USER_REQUEST = "依部門計算 Excel 中的平均延誤。"

# 不需要 Tool
USER_REQUEST = "請用一句話說明什麼是 Tool Calling。"

print("問題：", USER_REQUEST)
try:
    route = route_tool(USER_REQUEST)
    arguments = validate_tool_arguments(route)
    print("Pydantic 驗證後的路由：")
    show(route.model_dump())
    print("參數 Model：", type(arguments).__name__ if arguments else "none")
except Exception as error:
    print(f"路由失敗 [{type(error).__name__}]：{error}")

問題： 請用一句話說明什麼是 Tool Calling。
Pydantic 驗證後的路由：
{
  "tool_name": "none",
  "reason": "問題不需要查看圖片、執行數學運算或操作 Excel 表格，因此不需要使用工具。",
  "arguments": {}
}
參數 Model： none


## 執行路由後的 Math Tool

In [38]:
USER_REQUEST = "一架飛機有 312 個座位，載客率是 87%，精確計算旅客數"
route = route_tool(USER_REQUEST)
args = validate_tool_arguments(route)
if route.tool_name != "math_tool":
    raise RuntimeError(f"預期 math_tool，AI 卻選擇 {route.tool_name}：{route.reason}")
result = math_tool(args.a, args.op, args.b)
print_execution_trace(question=USER_REQUEST, tool=route.tool_name, tool_result=result, answer=f"計算結果：{result}")

AI Agent 執行追蹤
使用者問題：
一架飛機有 312 個座位，載客率是 87%，精確計算旅客數

是否需要工具：
math_tool

工具執行結果：
271.44

最終回答：
計算結果：271.44



## 執行路由後的 Image Tool

In [39]:
USER_REQUEST = "這張登機證是哪一個航班與登機門？"
route = route_tool(USER_REQUEST)
args = validate_tool_arguments(route)
if route.tool_name != "image_tool":
    raise RuntimeError(f"預期 image_tool，AI 卻選擇 {route.tool_name}：{route.reason}")
image_result = analyze_image(ROOT / "data/images/mock_boarding_pass.png", args.question)
show(image_result)
print_execution_trace(question=USER_REQUEST, tool=route.tool_name, tool_result=image_result["content"], answer=image_result["content"])

{
  "type": "image",
  "source": "mock_boarding_pass.png",
  "content": "根據這張登機證，航班號是 CI102，登機門是 B6。"
}
AI Agent 執行追蹤
使用者問題：
這張登機證是哪一個航班與登機門？

是否需要工具：
image_tool

工具執行結果：
根據這張登機證，航班號是 CI102，登機門是 B6。

最終回答：
根據這張登機證，航班號是 CI102，登機門是 B6。



## Excel Python Tool：固定程式與 AI 動態生成對照

In [40]:
EXCEL_PATH = ROOT / "data/excel/flight_delays.xlsx"

# A. 寫死的範例：可重現、方便解說 sandbox
FIXED_QUESTION = "找出延誤超過 120 分鐘的航班"
FIXED_CODE = """df = pd.read_excel(excel_path)
result = df.loc[df['delay_minutes'] > 120, ['flight','delay_minutes']].to_dict('records')"""
print("=== 固定程式 ===")
print(FIXED_CODE)
fixed_tool_result = safe_excel_python(EXCEL_PATH, FIXED_CODE)
show(fixed_tool_result)

=== 固定程式 ===
df = pd.read_excel(excel_path)
result = df.loc[df['delay_minutes'] > 120, ['flight','delay_minutes']].to_dict('records')
[
  {
    "flight": "CI102",
    "delay_minutes": 145
  },
  {
    "flight": "CI203",
    "delay_minutes": 180
  },
  {
    "flight": "CI407",
    "delay_minutes": 125
  }
]


## 固定案例的 AI Agent 執行追蹤

這個 Cell 只顯示本次執行追蹤，不與程式碼生成或其他測試輸出混在一起。

In [41]:
fixed_final_answer = ask_gpt_oss(
    FIXED_QUESTION,
    json.dumps(fixed_tool_result, ensure_ascii=False),
    "你是航空資料助理。只能根據 Tool 執行結果回答，不可補造資料。請用繁體中文簡潔回答。",
)
print_execution_trace(
    question=FIXED_QUESTION,
    tool="✓ excel_python_tool",
    tool_result=fixed_tool_result,
    answer=fixed_final_answer,
)

AI Agent 執行追蹤
使用者問題：
找出延誤超過 120 分鐘的航班

是否需要工具：
✓ excel_python_tool

工具執行結果：
[{'flight': 'CI102', 'delay_minutes': 145}, {'flight': 'CI203', 'delay_minutes': 180}, {'flight': 'CI407', 'delay_minutes': 125}]

最終回答：
延誤超過 120 分鐘的航班有：

1. CI102，延誤 145 分鐘
2. CI203，延誤 180 分鐘
3. CI407，延誤 125 分鐘



## AI 動態產生 Excel Python 程式碼：一次一題

每次只啟用一個 `DYNAMIC_QUESTION`，觀察 AI 產生的程式、理由與執行結果。

In [44]:
# 預設：分組平均
# DYNAMIC_QUESTION = "依 department 計算平均 delay_minutes。"

# 其他題目：請先註解上面的 DYNAMIC_QUESTION，再取消其中一行註解
# DYNAMIC_QUESTION = "找出延誤最久的航班、部門與延誤分鐘。"
DYNAMIC_QUESTION = "統計每個部門有幾個航班。"
# DYNAMIC_QUESTION = "列出延誤最久的前 3 個航班，並依延誤由高到低排序。"
# DYNAMIC_QUESTION = "同時回傳總旅客數、平均延誤與所有不重複部門。"

print("問題：", DYNAMIC_QUESTION)
try:
    route = route_tool(DYNAMIC_QUESTION)
    print("Tool Route："); show(route.model_dump())
    if route.tool_name != "excel_python_tool":
        raise RuntimeError(f"Router 選擇 {route.tool_name}：{route.reason}")
    validate_tool_arguments(route)
    generated = generate_excel_code(DYNAMIC_QUESTION, EXCEL_PATH)
    print("AI 生成理由：", generated.reason)
    print("AI 生成程式：\n", generated.code)
    dynamic_result = safe_excel_python(EXCEL_PATH, generated.code)
    print("執行結果：")
    show(dynamic_result)
except Exception as error:
    print(f"本題失敗 [{type(error).__name__}]：{error}")

問題： 統計每個部門有幾個航班。
Tool Route：
{
  "tool_name": "excel_python_tool",
  "reason": "問題需要對 Excel 表格進行統計和分組操作，以計算每個部門的航班數量，這符合 excel_python_tool 的使用描述。",
  "arguments": {
    "question": "統計每個部門有幾個航班。"
  }
}
AI 生成理由： 使用 value_counts() 計算每個部門的航班數量，並將結果轉換為字典格式。
AI 生成程式：
 df = pd.read_excel(excel_path)
result = df['department'].value_counts().to_dict()
執行結果：
{
  "航務部": 2,
  "地勤部": 2,
  "客服部": 1
}


## 將 Dynamic Tool Result 交回 LLM 包裝成最終回答

這個 Cell 只負責 final answer；上一格保留 generated code 與 pandas 原始結果。

In [45]:
if "dynamic_result" not in globals():
    raise RuntimeError("請先執行上一格，產生 dynamic_result。")

dynamic_final_answer = ask_gpt_oss(
    DYNAMIC_QUESTION,
    json.dumps(dynamic_result, ensure_ascii=False, default=str),
    "你是航空資料助理。只能根據 Tool 執行結果回答，不可補造、重新計算或改變數值。請用繁體中文清楚摘要。",
)
print_execution_trace(
    question=DYNAMIC_QUESTION,
    tool="✓ excel_python_tool (AI generated code)",
    tool_result=dynamic_result,
    answer=dynamic_final_answer,
)

AI Agent 執行追蹤
使用者問題：
統計每個部門有幾個航班。

是否需要工具：
✓ excel_python_tool (AI generated code)

工具執行結果：
{'航務部': 2, '地勤部': 2, '客服部': 1}

最終回答：
根據提供的內容，統計如下：

- 航務部有 2 個航班
- 地勤部有 2 個航班
- 客服部有 1 個航班

